# Local LLM Setup: The Ollama Guide

Running LLMs locally gives you **three superpowers** that cloud APIs can't match:
1. **Total Privacy** — Your data never leaves your machine. Critical for sensitive/proprietary work.
2. **Zero Cost** — No API bills, no rate limits, no monthly subscriptions.
3. **No Latency** — No network round-trips. Responses start instantly.

The easiest way to run local LLMs today is **Ollama** — it handles model downloading, quantization, and GPU acceleration automatically.

## 1. Installation

1. **Download**: Go to [ollama.com](https://ollama.com) and download the version for your OS (Windows, macOS, or Linux).
2. **Install**: Run the installer — it's a standard app installation.
3. **Verify**: Once installed, Ollama runs as a background service. You should see a small llama icon in your system tray (Windows) or menu bar (macOS).

### How Ollama Works Under the Hood

Ollama is actually a thin wrapper around **llama.cpp**, a C++ library that runs quantized models efficiently. When you `ollama run llama3`, here's what actually happens:

```
1. Download GGUF model file → ~/.ollama/models/
2. Load model weights into RAM (and VRAM if GPU is available)
3. Start an HTTP server on localhost:11434
4. Accept prompts via API → process through model → return tokens
```

That HTTP server is what the Python library (`ollama`) talks to behind the scenes.

## 2. Let's Verify Your Installation

Let's check that Ollama is running and see what models you have available.

In [ ]:
# Check if Ollama is running and list installed models
!ollama list

In [ ]:
# If the above shows nothing or errors, you need to pull a model first.
# Uncomment the line below to download Llama 3 (~4.7GB download):

# !ollama pull llama3

In [ ]:
# Let's verify with Python too
import ollama

try:
    models = ollama.list()
    print("✅ Ollama is running!\n")
    print("Installed models:")
    for model in models.get('models', []):
        name = model.get('name', 'unknown')
        size_gb = model.get('size', 0) / (1024**3)
        print(f"  • {name} ({size_gb:.1f} GB)")
except Exception as e:
    print(f"❌ Ollama is not running! Error: {e}")
    print("\nMake sure Ollama is running:")
    print("  1. Check your system tray for the Ollama icon")
    print("  2. Or open a terminal and run: ollama serve")

## 3. Pulling Your First Model

In the terminal, just run:

```bash
ollama run llama3
```

This does two things at once: **downloads** the model (first time only) and **starts a chat**. Type your message and press Enter to chat. Type `/bye` to exit.

### What's Happening During Download?

When you pull a model, Ollama downloads a **GGUF file** — that's the quantized model weights. For Llama 3 (8B, Q4_K_M), it's about 4.7GB. The model is stored in:
- **Windows**: `C:\Users\<you>\.ollama\models\`
- **macOS/Linux**: `~/.ollama/models/`

## 4. Recommended Models (2024-2025)

Here's a practical guide based on your hardware:

### If you have 8GB RAM (minimum)

| Model | Command | Size | Best For |
| :--- | :--- | :--- | :--- |
| **Phi-3 Mini** | `ollama run phi3:mini` | ~2.3GB | Fast, runs on anything |
| **Gemma 2 (2B)** | `ollama run gemma2:2b` | ~1.6GB | Google's tiny powerhouse |
| **Qwen 2.5 (3B)** | `ollama run qwen2.5:3b` | ~2.0GB | Great multilingual support |

### If you have 16GB RAM (recommended)

| Model | Command | Size | Best For |
| :--- | :--- | :--- | :--- |
| **Llama 3.1 (8B)** | `ollama run llama3.1` | ~4.7GB | Best all-rounder |
| **Mistral (7B)** | `ollama run mistral` | ~4.1GB | Fast and efficient |
| **DeepSeek-Coder** | `ollama run deepseek-coder` | ~4.7GB | Specialized for code |
| **Llama 3.2 (3B)** | `ollama run llama3.2` | ~2.0GB | Newest, very capable for size |

### If you have 32GB+ RAM or a good GPU

| Model | Command | Size | Best For |
| :--- | :--- | :--- | :--- |
| **Llama 3.1 (70B)** | `ollama run llama3.1:70b` | ~40GB | Near-GPT-4 quality |
| **Mixtral (8x7B)** | `ollama run mixtral` | ~26GB | MoE architecture |
| **DeepSeek-V2** | `ollama run deepseek-v2` | ~16GB | Excellent coding AI |

## 5. Model Management

Here are the essential Ollama commands you'll use regularly:

In [1]:
# Show detailed info about a model
!ollama show llama3

  Model
    architecture        llama    
    parameters          8.0B     
    context length      8192     
    embedding length    4096     
    quantization        Q4_0     

  Capabilities
    completion    

  Parameters
    num_keep    24                       
    stop        "<|start_header_id|>"    
    stop        "<|end_header_id|>"      
    stop        "<|eot_id|>"             

  License
    META LLAMA 3 COMMUNITY LICENSE AGREEMENT             
    Meta Llama 3 Version Release Date: April 18, 2024    
    ...                                                  



In [2]:
# Useful commands reference (don't run all of these — just for reference):
print("""
Essential Ollama Commands:
─────────────────────────
ollama list            → See all installed models
ollama pull <model>    → Download a model (without starting a chat)
ollama run <model>     → Download (if needed) + start chatting
ollama show <model>    → See model details (size, parameters, template)
ollama rm <model>      → Delete a model to free disk space
ollama ps              → See currently loaded models (using RAM/VRAM)
ollama serve           → Start the Ollama server manually
""")


Essential Ollama Commands:
─────────────────────────
ollama list            → See all installed models
ollama pull <model>    → Download a model (without starting a chat)
ollama run <model>     → Download (if needed) + start chatting
ollama show <model>    → See model details (size, parameters, template)
ollama rm <model>      → Delete a model to free disk space
ollama ps              → See currently loaded models (using RAM/VRAM)
ollama serve           → Start the Ollama server manually



## 6. Hardware Tips: Getting the Best Performance

### RAM is King
The model weights need to fit entirely in RAM (or VRAM). If they don't fit, the model will use disk swap and become *painfully* slow.

**Quick math:** A Q4-quantized model needs roughly `parameters × 0.6 GB` of RAM:
- 3B model → ~2 GB
- 8B model → ~5 GB
- 70B model → ~40 GB

### GPU Acceleration
If you have an **NVIDIA GPU** or **Apple Silicon** (M1/M2/M3), Ollama will automatically use it. This gives you **3-10x faster** token generation.

To check if GPU is being used:

In [3]:
# Check if NVIDIA GPU is available
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>nul || echo "No NVIDIA GPU detected (that's okay, Ollama will use CPU)."

"No NVIDIA GPU detected (that's okay, Ollama will use CPU)."


## 7. Quick Benchmark: How Fast is Your Setup?

In [4]:
import ollama
import time

prompt = "Explain what remote sensing is in exactly 3 sentences."

start = time.time()
response = ollama.generate(model='llama3', prompt=prompt)
elapsed = time.time() - start

# Rough token count (response only)
words = len(response['response'].split())
approx_tokens = int(words * 1.33)  # rough estimate

print(response['response'])
print(f"\n--- Performance ---")
print(f"Time: {elapsed:.1f} seconds")
print(f"Approx tokens: {approx_tokens}")
print(f"Speed: ~{approx_tokens/elapsed:.1f} tokens/sec")

Remote sensing is the acquisition of information about the Earth's surface through indirect methods, such as aerial photography or satellite imaging, without physically being present at the location. This technology allows for the collection of data on various aspects of the environment, including land cover, soil moisture, and atmospheric conditions. By analyzing these data sets, scientists can gain insights into natural phenomena, monitor environmental changes, and support decision-making in fields like agriculture, conservation, and urban planning.

--- Performance ---
Time: 66.9 seconds
Approx tokens: 101
Speed: ~1.5 tokens/sec


**Performance guide:**
- **< 5 tokens/sec**: CPU-only, larger model — usable but slow
- **10-30 tokens/sec**: Good CPU or partial GPU offload — comfortable for interactive use
- **30-100+ tokens/sec**: Good GPU acceleration — snappy responses

## 8. Troubleshooting

### `ConnectionError: Failed to connect to Ollama`
The Ollama server isn't running.
1. Look for the Ollama icon in your system tray (Windows) or menu bar (Mac)
2. If it's not there, search for "Ollama" in your Start Menu and launch it
3. Or run `ollama serve` in a terminal

### `Model not found`
You haven't downloaded the model yet.
1. Run `ollama pull <model_name>` in a terminal
2. Check spelling — it's `llama3` not `llama-3`

### Very slow generation
1. Check if you're running out of RAM (model is swapping to disk)
2. Try a smaller model (`phi3:mini` or `gemma2:2b`)
3. Close other memory-hungry apps

---

**Next notebook:** [LLM Inference with Python](./llm_inference_python.ipynb) — Now that Ollama is running, let's interact with it programmatically!